# Gold: Citi Bike Station Activity (v1)

Reads the `citibike.tripdata` bronze table (see
`data-generators/bronze_ingest_citibike.ipynb`) and aggregates it by start
station: total rides, member vs. casual split, and average trip duration.
Persists the result as a managed Spark table for demo dashboards/queries.

Re-runnable: `mode("overwrite")`, so re-running just refreshes the table.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

## Configuration

In [ ]:
DATABASE_NAME = "citibike"
SOURCE_TABLE = "tripdata"
TARGET_TABLE = "gold_station_activity_v1"

## Load bronze table

In [ ]:
df = spark.table(f"{DATABASE_NAME}.{SOURCE_TABLE}")

## Transform: station activity summary

Aggregates by start station: total rides, member vs. casual ride counts,
and average trip duration in minutes (`ended_at` - `started_at`).

In [ ]:
summary_df = (
    df
    .filter(F.col("start_station_name").isNotNull())
    .withColumn(
        "duration_minutes",
        (F.unix_timestamp("ended_at") - F.unix_timestamp("started_at")) / 60.0,
    )
    .groupBy("start_station_name")
    .agg(
        F.count("*").alias("total_rides"),
        F.sum(F.when(F.col("member_casual") == "member", 1).otherwise(0)).alias("member_rides"),
        F.sum(F.when(F.col("member_casual") == "casual", 1).otherwise(0)).alias("casual_rides"),
        F.round(F.avg("duration_minutes"), 2).alias("avg_duration_minutes"),
    )
    .orderBy(F.desc("total_rides"))
)

## Persist as a managed Spark table

In [ ]:
summary_df.write \
    .mode("overwrite") \
    .saveAsTable(f"{DATABASE_NAME}.{TARGET_TABLE}")

## Verify

In [ ]:
spark.sql(f"SELECT COUNT(*) AS total_stations FROM {DATABASE_NAME}.{TARGET_TABLE}").show()
spark.sql(f"SELECT * FROM {DATABASE_NAME}.{TARGET_TABLE} ORDER BY total_rides DESC LIMIT 10").show()